<a href="https://colab.research.google.com/github/deekshdechamma/DL_skill_developer/blob/main/dl6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 6: Bidirectional Long Short-Term Memory (BiLSTM) Cell
Mechanics from Scratch
● Objective: Learn sequence transduction architectures and dynamic temporal tracking by
implementing gated recurrent equations and memory routing layouts from scratch.
● Required Tech Stack: PyTorch (Tensor operations only, no nn.RNN or nn.LSTM), NumPy.
● Task Description: Students will design and execute a bidirectional LSTM layer. They must
write individual loop-based step updates calculating Input ( ), Forget ( ), Output ( ),
and Cell State Candidate ( ) activations, stitch backward and forward sequence steps,
and handle dynamic sequence-padded packs.

In [1]:
#Step 1 — Import libraries
import torch
import numpy as np

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Autograd is not used
torch.set_grad_enabled(False)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


In [2]:
#Step 2 — Define activation functions
def sigmoid(x):
    return 1.0 / (1.0 + torch.exp(-x))

def tanh(x):
    return torch.tanh(x)

In [3]:
#Step 3 — Create an LSTM cell from scratch
class LSTMCellScratch:

    def __init__(self, input_size, hidden_size):

        self.input_size = input_size
        self.hidden_size = hidden_size

        # Xavier-like initialization
        scale = np.sqrt(2.0 / (input_size + hidden_size))

        # Input gate
        self.W_xi = torch.randn(input_size, hidden_size) * scale
        self.W_hi = torch.randn(hidden_size, hidden_size) * scale
        self.b_i = torch.zeros(hidden_size)

        # Forget gate
        self.W_xf = torch.randn(input_size, hidden_size) * scale
        self.W_hf = torch.randn(hidden_size, hidden_size) * scale
        self.b_f = torch.ones(hidden_size)

        # Output gate
        self.W_xo = torch.randn(input_size, hidden_size) * scale
        self.W_ho = torch.randn(hidden_size, hidden_size) * scale
        self.b_o = torch.zeros(hidden_size)

        # Cell-state candidate
        self.W_xg = torch.randn(input_size, hidden_size) * scale
        self.W_hg = torch.randn(hidden_size, hidden_size) * scale
        self.b_g = torch.zeros(hidden_size)


    def forward(self, x_t, h_prev, c_prev):

        # Input gate
        i_t = sigmoid(
            x_t @ self.W_xi +
            h_prev @ self.W_hi +
            self.b_i
        )

        # Forget gate
        f_t = sigmoid(
            x_t @ self.W_xf +
            h_prev @ self.W_hf +
            self.b_f
        )

        # Output gate
        o_t = sigmoid(
            x_t @ self.W_xo +
            h_prev @ self.W_ho +
            self.b_o
        )

        # Candidate cell state
        g_t = tanh(
            x_t @ self.W_xg +
            h_prev @ self.W_hg +
            self.b_g
        )

        # New cell state
        c_t = f_t * c_prev + i_t * g_t

        # New hidden state
        h_t = o_t * tanh(c_t)

        return h_t, c_t, i_t, f_t, o_t, g_t

In [5]:
#Step 4 — Create the BiLSTM layer
class BiLSTMScratch:

    def __init__(self, input_size, hidden_size):

        self.input_size = input_size
        self.hidden_size = hidden_size

        # Forward-direction cell
        self.forward_cell = LSTMCellScratch(
            input_size,
            hidden_size
        )

        # Backward-direction cell
        self.backward_cell = LSTMCellScratch(
            input_size,
            hidden_size
        )

In [6]:
#Step 5 — Add forward sequence processing
def forward_direction(self, x, lengths):

    batch_size, seq_len, _ = x.shape

    h = torch.zeros(batch_size, self.hidden_size)
    c = torch.zeros(batch_size, self.hidden_size)

    outputs = []

    for t in range(seq_len):

        x_t = x[:, t, :]

        h_new, c_new, i, f, o, g = self.forward_cell.forward(
            x_t, h, c
        )

        # Check which sequences are still active
        mask = (t < lengths).float().unsqueeze(1)

        h = h_new * mask + h * (1 - mask)
        c = c_new * mask + c * (1 - mask)

        # Padding positions output zero
        outputs.append(h * mask)

    return torch.stack(outputs, dim=1)

In [7]:
#Step 6 — Backward sequence processing
def backward_direction(self, x, lengths):

    batch_size, seq_len, _ = x.shape

    h = torch.zeros(batch_size, self.hidden_size)
    c = torch.zeros(batch_size, self.hidden_size)

    outputs = [
        torch.zeros(batch_size, self.hidden_size)
        for _ in range(seq_len)
    ]

    for t in reversed(range(seq_len)):

        x_t = x[:, t, :]

        mask = (t < lengths).float().unsqueeze(1)

        h_new, c_new, i, f, o, g = self.backward_cell.forward(
            x_t, h, c
        )

        h = h_new * mask + h * (1 - mask)
        c = c_new * mask + c * (1 - mask)

        outputs[t] = h * mask

    return torch.stack(outputs, dim=1)

In [8]:
#Step 7 — Complete BiLSTM class
class BiLSTMScratch:

    def __init__(self, input_size, hidden_size):

        self.input_size = input_size
        self.hidden_size = hidden_size

        self.forward_cell = LSTMCellScratch(
            input_size,
            hidden_size
        )

        self.backward_cell = LSTMCellScratch(
            input_size,
            hidden_size
        )


    def forward_direction(self, x, lengths):

        batch_size, seq_len, _ = x.shape

        h = torch.zeros(batch_size, self.hidden_size)
        c = torch.zeros(batch_size, self.hidden_size)

        outputs = []

        for t in range(seq_len):

            x_t = x[:, t, :]

            h_new, c_new, _, _, _, _ = \
                self.forward_cell.forward(x_t, h, c)

            mask = (t < lengths).float().unsqueeze(1)

            h = h_new * mask + h * (1 - mask)
            c = c_new * mask + c * (1 - mask)

            outputs.append(h * mask)

        return torch.stack(outputs, dim=1)


    def backward_direction(self, x, lengths):

        batch_size, seq_len, _ = x.shape

        h = torch.zeros(batch_size, self.hidden_size)
        c = torch.zeros(batch_size, self.hidden_size)

        outputs = [
            torch.zeros(batch_size, self.hidden_size)
            for _ in range(seq_len)
        ]

        for t in reversed(range(seq_len)):

            x_t = x[:, t, :]

            mask = (t < lengths).float().unsqueeze(1)

            h_new, c_new, _, _, _, _ = \
                self.backward_cell.forward(x_t, h, c)

            h = h_new * mask + h * (1 - mask)
            c = c_new * mask + c * (1 - mask)

            outputs[t] = h * mask

        return torch.stack(outputs, dim=1)


    def forward(self, x, lengths):

        # Forward LSTM
        forward_output = self.forward_direction(
            x,
            lengths
        )

        # Backward LSTM
        backward_output = self.backward_direction(
            x,
            lengths
        )

        # Concatenate both directions
        bilstm_output = torch.cat(
            [forward_output, backward_output],
            dim=2
        )

        return bilstm_output

In [9]:
#Step 8 — Create dynamically padded sequences
input_size = 4
hidden_size = 6

lengths = torch.tensor([5, 3, 4])

batch_size = len(lengths)
max_length = int(lengths.max())

x = torch.zeros(
    batch_size,
    max_length,
    input_size
)

for i, length in enumerate(lengths):
    x[i, :length] = torch.randn(
        length,
        input_size
    )

print("Input shape:", x.shape)
print("\nSequence lengths:")
print(lengths)

print("\nPadded input:")
print(x)

Input shape: torch.Size([3, 5, 4])

Sequence lengths:
tensor([5, 3, 4])

Padded input:
tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055],
         [-0.7581,  1.0783,  0.8008,  1.6806],
         [ 0.3559, -0.6866, -0.4934,  0.2415],
         [-0.2316,  0.0418, -0.2516,  0.8599],
         [-0.3097, -0.3957,  0.8034, -0.6216]],

        [[-0.7658, -0.7506,  1.3525,  0.6863],
         [-0.3278,  0.7950,  0.2815,  0.0562],
         [ 0.5227, -0.2384, -0.0499,  0.5263],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000]],

        [[ 0.3672,  0.1754,  1.3852, -0.4459],
         [ 1.4451,  0.8564,  2.2181,  0.5232],
         [ 1.1754,  0.5612, -0.4527, -0.7718],
         [-0.1722,  0.5238,  0.0566,  0.4263],
         [ 0.0000,  0.0000,  0.0000,  0.0000]]])


In [10]:
#Step 9 — Create the custom BiLSTM
bilstm = BiLSTMScratch(
    input_size=input_size,
    hidden_size=hidden_size
)

print("BiLSTM created successfully!")

BiLSTM created successfully!


In [11]:
#Step 10 — Perform the forward pass
output = bilstm.forward(
    x,
    lengths
)

print("BiLSTM output shape:")
print(output.shape)

BiLSTM output shape:
torch.Size([3, 5, 12])


In [12]:
#Step 11 — Display the BiLSTM output
print("\nComplete BiLSTM Output:\n")
print(output)


Complete BiLSTM Output:

tensor([[[-0.1254,  0.0161,  0.0503, -0.2742,  0.0398, -0.2504,  0.0636,
          -0.0328,  0.5616, -0.0888,  0.4103,  0.2660],
         [ 0.0664,  0.2754, -0.0723,  0.0271,  0.4107,  0.2043, -0.2662,
           0.0221,  0.0186, -0.1897,  0.3541, -0.0142],
         [ 0.0633,  0.1629,  0.0972, -0.1400,  0.1374,  0.1883, -0.0384,
          -0.0014,  0.0202, -0.1695, -0.0537, -0.0185],
         [ 0.1719,  0.1602, -0.0215, -0.0350,  0.2584,  0.3087, -0.1226,
          -0.0114,  0.0189, -0.2111, -0.0533, -0.0246],
         [ 0.1918, -0.0220,  0.0471, -0.0991,  0.0341,  0.0752, -0.0378,
          -0.0884,  0.0629,  0.0837, -0.2587, -0.0056]],

        [[ 0.2449, -0.2644, -0.1332, -0.0315, -0.0809,  0.1670, -0.1244,
           0.0224,  0.0311, -0.0545,  0.1142, -0.0124],
         [ 0.0969, -0.0495, -0.3006,  0.1141,  0.1373,  0.1622, -0.0012,
           0.0525,  0.0128, -0.0868,  0.2602,  0.0094],
         [ 0.0504, -0.0588, -0.3024, -0.0818,  0.1039,  0.2372,  0.03

In [13]:
for i in range(batch_size):

    print("\n==========================")
    print("Sequence:", i + 1)
    print("Actual Length:", lengths[i].item())
    print("==========================")

    print(output[i])


Sequence: 1
Actual Length: 5
tensor([[-0.1254,  0.0161,  0.0503, -0.2742,  0.0398, -0.2504,  0.0636, -0.0328,
          0.5616, -0.0888,  0.4103,  0.2660],
        [ 0.0664,  0.2754, -0.0723,  0.0271,  0.4107,  0.2043, -0.2662,  0.0221,
          0.0186, -0.1897,  0.3541, -0.0142],
        [ 0.0633,  0.1629,  0.0972, -0.1400,  0.1374,  0.1883, -0.0384, -0.0014,
          0.0202, -0.1695, -0.0537, -0.0185],
        [ 0.1719,  0.1602, -0.0215, -0.0350,  0.2584,  0.3087, -0.1226, -0.0114,
          0.0189, -0.2111, -0.0533, -0.0246],
        [ 0.1918, -0.0220,  0.0471, -0.0991,  0.0341,  0.0752, -0.0378, -0.0884,
          0.0629,  0.0837, -0.2587, -0.0056]])

Sequence: 2
Actual Length: 3
tensor([[ 0.2449, -0.2644, -0.1332, -0.0315, -0.0809,  0.1670, -0.1244,  0.0224,
          0.0311, -0.0545,  0.1142, -0.0124],
        [ 0.0969, -0.0495, -0.3006,  0.1141,  0.1373,  0.1622, -0.0012,  0.0525,
          0.0128, -0.0868,  0.2602,  0.0094],
        [ 0.0504, -0.0588, -0.3024, -0.0818,  0.10

In [14]:
#Step 12 — Inspect individual LSTM gates
x_t = x[:, 0, :]

h_prev = torch.zeros(
    batch_size,
    hidden_size
)

c_prev = torch.zeros(
    batch_size,
    hidden_size
)

h_t, c_t, i_t, f_t, o_t, g_t = \
    bilstm.forward_cell.forward(
        x_t,
        h_prev,
        c_prev
    )

print("INPUT GATE i_t:")
print(i_t)

print("\nFORGET GATE f_t:")
print(f_t)

print("\nOUTPUT GATE o_t:")
print(o_t)

print("\nCELL CANDIDATE g_t:")
print(g_t)

print("\nCELL STATE c_t:")
print(c_t)

print("\nHIDDEN STATE h_t:")
print(h_t)

INPUT GATE i_t:
tensor([[0.2701, 0.2977, 0.8484, 0.8309, 0.6216, 0.3408],
        [0.4929, 0.6990, 0.3728, 0.2776, 0.4765, 0.7732],
        [0.3792, 0.5455, 0.6032, 0.5197, 0.5448, 0.6398]])

FORGET GATE f_t:
tensor([[0.8112, 0.8169, 0.9365, 0.7088, 0.3788, 0.7750],
        [0.4990, 0.7627, 0.6353, 0.8883, 0.8629, 0.9252],
        [0.6196, 0.7902, 0.8165, 0.8503, 0.7167, 0.9001]])

OUTPUT GATE o_t:
tensor([[0.5016, 0.0657, 0.1548, 0.7033, 0.4125, 0.7660],
        [0.6641, 0.5483, 0.5168, 0.3181, 0.5685, 0.3812],
        [0.6185, 0.2577, 0.3326, 0.4691, 0.5136, 0.5591]])

CELL CANDIDATE g_t:
tensor([[-0.9456,  0.8386,  0.3978, -0.4954,  0.1557, -0.9955],
        [ 0.7851, -0.7522, -0.7075, -0.3573, -0.3007,  0.6076],
        [-0.0679, -0.1067, -0.4470, -0.4685, -0.1155, -0.7070]])

CELL STATE c_t:
tensor([[-0.2554,  0.2497,  0.3375, -0.4117,  0.0968, -0.3393],
        [ 0.3870, -0.5258, -0.2638, -0.0992, -0.1433,  0.4698],
        [-0.0257, -0.0582, -0.2696, -0.2435, -0.0629, -0.4524]])

In [15]:
#Step 13 — Verify gate ranges
print("Input gate range:")
print(i_t.min().item(), i_t.max().item())

print("\nForget gate range:")
print(f_t.min().item(), f_t.max().item())

print("\nOutput gate range:")
print(o_t.min().item(), o_t.max().item())

print("\nCandidate range:")
print(g_t.min().item(), g_t.max().item())

Input gate range:
0.27010631561279297 0.8484134078025818

Forget gate range:
0.37880054116249084 0.9364897608757019

Output gate range:
0.06574051082134247 0.7659918665885925

Candidate range:
-0.995538055896759 0.8385549187660217


In [17]:
#Step 14 — Show forward and backward outputs separately
forward_output = bilstm.forward_direction(
    x,
    lengths
)

backward_output = bilstm.backward_direction(
    x,
    lengths
)

print("Forward output shape:")
print(forward_output.shape)

print("\nBackward output shape:")
print(backward_output.shape)

print("\nCombined BiLSTM shape:")
print(output.shape)

Forward output shape:
torch.Size([3, 5, 6])

Backward output shape:
torch.Size([3, 5, 6])

Combined BiLSTM shape:
torch.Size([3, 5, 12])


In [18]:
#Step 15 — Verify dynamic padding
for batch_index in range(batch_size):

    actual_length = lengths[batch_index].item()

    print(
        f"Sequence {batch_index + 1} "
        f"- Actual Length = {actual_length}"
    )

    if actual_length < max_length:

        padded_output = output[
            batch_index,
            actual_length:
        ]

        print("Padded output:")
        print(padded_output)

        print(
            "Padding correctly zero:",
            torch.all(padded_output == 0).item()
        )

    print()

Sequence 1 - Actual Length = 5

Sequence 2 - Actual Length = 3
Padded output:
tensor([[0., -0., -0., -0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., -0., -0., -0., 0., 0., 0., 0., 0., 0., 0., 0.]])
Padding correctly zero: True

Sequence 3 - Actual Length = 4
Padded output:
tensor([[-0., 0., -0., 0., 0., -0., 0., 0., 0., 0., 0., 0.]])
Padding correctly zero: True

